In [28]:
import os
import pandas as pd
import requests

In [29]:
# Set up headers with token
API_TOKEN = os.getenv("football-data-token")

if not API_TOKEN:
    raise ValueError("football-data-token environment variable is not set.")

HEADERS = {
    "X-Auth-Token": API_TOKEN
}

In [30]:
def send_get_request(url, headers=HEADERS):
    # Send the GET request
    response = requests.get(url=url, headers=headers)

    # Return processed data
    return response.json()


def get_current_matchday():
    url = "https://api.football-data.org/v4/competitions/PL"
    data = send_get_request(url, HEADERS)
    return data["currentSeason"]["currentMatchday"]


def load_predictions(m):
    cwd = os.getcwd()
    preds = None
    for f in os.listdir(f"{cwd}/predictions"):
        if int(f.split("_")[1]) == m:
            preds = pd.read_csv(f"predictions/{f}")
    
    if preds is None:
        print("Predictions not found!")
    
    return preds


def get_matchday_results(num):
    # Define matchday endpoint based on input
    url = f"https://api.football-data.org/v4/competitions/PL/matches?matchday={num}"
    data = send_get_request(url, HEADERS)

    results = pd.DataFrame(
        [{
            "home": m["homeTeam"]["name"],
            "away": m["awayTeam"]["name"],
            "winner": m["score"]["winner"],
            "home_goals": m["score"]["fullTime"]["home"],
            "away_goals": m["score"]["fullTime"]["away"],
            
        } for m in data["matches"]]
    )

    return results

In [ ]:
MATCHDAY = get_current_matchday()
print(f"It's matchday {MATCHDAY}!")

In [ ]:
# Load predicted scores
preds_df = load_predictions(MATCHDAY)
preds_df

In [ ]:
# Fetch actual results
actuals_df = get_matchday_results(MATCHDAY)
actuals_df

In [34]:
# Write actuals to CSV
actuals_df.to_csv(f"./actuals/matchday_{MATCHDAY}_actuals.csv", index=False)

In [35]:
with open(f"./eval_output/eval_{MATCHDAY}.txt", "w") as f:
    for i in range(actuals_df.shape[0]):
        symbol = "✅" if preds_df.loc[i, "winner"] == actuals_df.loc[i, "winner"] else "❌"
        match_title = f'{actuals_df.loc[i, "away"]} @ {actuals_df.loc[i, "home"]}'
        predicted_score = f'{preds_df.loc[i, "home"]} [{preds_df.loc[i, "home_goals"]} - {preds_df.loc[i, "away_goals"]}] {preds_df.loc[i, "away"]}'
        actual_score = f'{actuals_df.loc[i, "home"]} [{actuals_df.loc[i, "home_goals"]} - {actuals_df.loc[i, "away_goals"]}] {actuals_df.loc[i, "away"]}'

        f.write(f"Prediction: {predicted_score} {symbol}\n")
        f.write(f"Actual: {actual_score}\n")
        f.write("\n")